# nb02 — Reduced metric panel: central clustermap (Figure 3) and Supplementary Figures S4, S5, S6

**Plan:** `article_1_revision_implementation_plan_261029.md`, WP9 §9.1 (Figure 3) and §9.2
(new S4 = old S6, new S5 = old S7, new S6 = old S9).

**Status.** The reduced panel (`output/wp2/reduced_panel.tsv`, 22 metrics) was approved at gate G1
(2026-11-03). The selection rule was frozen (G2, 2026-11-03) and the visual check closed (G2b,
2026-11-07): Figure 3 marks the **22 recommended approaches** (rule front, accepted at the visual
check; `output/wp4/selection_after_visual_check.tsv`). This notebook never computes a front.

**Gate G4 (Daniil, 2026-11-07).** No mark of the submitted version (no comparison with the
pre-revision result anywhere). The row and column clusters are colour-coded and named, as in the
original clustermap Daniil assembled in Figma (`figures_for_article/current_figures_tables_for_article_260915/Figure 3.jpg`):
shaded bands with names over the column dendrogram and beside the row dendrogram, black lines
between the clusters on the heat map. Cluster names are derived from the data (rule in §2b).

| Input | Content |
|---|---|
| `output/wp2/reduced_panel_matrix.parquet` | 2,234 approaches × 22 panel metrics, polarity-adjusted, min-max scaled (1 = best) |
| `output/wp2/reduced_panel.tsv` | panel metrics: metric group, annotation column, score bar |
| `output/wp2/cluster_membership.tsv` | the 33 candidate metrics with their correlation cluster |
| `output/wp2/metric_abs_spearman.parquet` | 33 × 33 \|Spearman ρ\| over the 2,234 approaches |
| `output/wp2/coverage_additions.tsv` | metrics added back by the coverage rule |
| `inputs/Supplementary File 3.csv.gz`, `inputs/Supplementary File 2.xlsx` | 87 published metrics, polarity |
| `output/wp4/selection_after_visual_check.tsv` | the 22 recommended approaches (gate G2b closed) |

**Outputs** (`figures_svg/`, SVG + PDF + PNG 200 dpi each): `fig3_reduced_panel_clustermap`,
`figS4_metric_availability`, `figS5_metric_correlation_clusters`, `figS6_approach_correlation`;
tables `output/wp9/nb02_*.tsv`.

**Run:**
`jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=collagen_3_11 nb02_reduced_panel_clustermap.ipynb`
(from `article_1_revision/`).

## §0 Configuration

### §0a Imports

In [ ]:
# Imports. revision_lib is used only for plain-word names and paths (plan WP9).
import json
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

import revision_lib as rl

### §0b Paths and switches

| Switch | Default | Meaning |
|---|---|---|
| `NA_FILL_FOR_CLUSTERING` | `"median"` | 241 of 2,234 approaches miss ≥ 1 panel metric. For the dendrogram only, a missing value is replaced by the column median; the heat map shows these cells in black (as in the first article version). `"drop"` instead clusters only the 1,993 complete approaches (as the submitted Figure 3 did with `dropna`). |
| `CLUSTER_METHOD` | `"ward"` | linkage of the approach and metric dendrograms of Figure 3 / S6 (as in the submitted figures) |
| `N_COLUMN_CLUSTERS`, `N_ROW_CLUSTERS` | 4, 5 | number of named approach (column) and metric (row) clusters cut from the Ward dendrograms of Figure 3 (the original figure had 4 approach and 6 metric clusters) |
| `PANEL_CUT` | `0.2` | distance cut 1 − \|ρ\| of the WP2 rule (\|ρ\| ≥ 0.8), drawn on the S5 dendrogram |

In [ ]:
# Paths and switches.
REVISION_DIR = rl.REVISION_DIR
WP2_DIR = rl.OUTPUT_DIR / "wp2"
RECOMMENDED_PATH = rl.OUTPUT_DIR / "wp4/selection_after_visual_check.tsv"
SUPP_FILE_2_PATH = rl.INPUTS_DIR / "Supplementary File 2.xlsx"
SUPP_FILE_3_PATH = rl.INPUTS_DIR / "Supplementary File 3.csv.gz"
FIGURES_DIR = REVISION_DIR / "figures_svg"
TABLES_DIR = rl.OUTPUT_DIR / "wp9"
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

NA_FILL_FOR_CLUSTERING = "median"  # "median" or "drop"
CLUSTER_METHOD = "ward"
PANEL_CUT = 0.2
N_COLUMN_CLUSTERS = 4
N_ROW_CLUSTERS = 5
APPROACH_KEYS = ["strat", "imp", "method", "post_rm"]

### §0c Palettes and plain-word names

Strategy, imputation and method palettes are copied from the frozen
`figures_for_article/Finally_assembled_figures_for_article.ipynb` (cell 4), so that the colours
match the submitted Figure 3. The score-bar and annotation-column colours follow the submitted
`annot_pal` (batch side brown, diagnosis side slate). Every label comes from the plain-word
dictionaries of `revision_lib` (plan §1.1). Metric groups use the pastel Set3 palette over the letters A–K and the column bars include
post-removal and method harshness, as in the submitted notebook and the Figma Figure 3 (Daniil,
G4 review 2026-11-08). Metric group letters A–K are kept because they are
published in Supplementary File 2; each is followed by its plain name.

In [ ]:
# Palettes copied from the frozen figure notebook (cell 4) and plain-word labels.
strat_pal = {
    "S0_no_removal": (0.631, 0.788, 0.957),
    "A_confirmed_bad": (1.000, 0.706, 0.510),
    "B_extended_bad": (0.553, 0.898, 0.631),
    "C_rnaseq_only": (1.000, 0.624, 0.608),
    "D_malignant_only": (0.816, 0.733, 1.000),
    "E1_iterative_r1": (0.871, 0.733, 0.608),
    "E2_iterative_r2": (0.980, 0.690, 0.894),
    "E3_iterative_r3": (0.812, 0.812, 0.812),
    "F_microarray_only": (1.000, 0.996, 0.639),
    "G_affymetrix_only": (0.725, 0.949, 0.941),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only": "#7ecbc4",
    "K_ffpe_only": "#f2c27f",
}
imp_pal = {"strict": "#2979ae", "knn": "#982d22", "softimpute": "#713689"}
IMPUTATION_NAMES = {"strict": "No imputation", "knn": "KNN", "softimpute": "softImpute"}

GROUP_NAMES = {  # published group letters + plain names (plan §0)
    "A": "A · PCA variance",
    "B": "B · Neighbour-based",
    "C": "C · Embedding",
    "D": "D · Distribution",
    "E": "E · Data quality",
    "G": "G · Graph connectivity",
    "H": "H · Pairwise distances",
    "I": "I · WaterMelon",
    "K": "K · Missing-value retention",
}
# Metric groups: the pastel Set3 palette over the letters A–K of the submitted figure notebook
# (group_pal, cell 4) and the Figma Figure 3 (node 403-22), so a letter keeps its colour
# (Daniil, G4 review 2026-11-08).
_set3 = sns.color_palette("Set3", n_colors=11).as_hex()
group_pal = {letter: _set3["ABCDEFGHIJK".index(letter)] for letter in GROUP_NAMES}

# Post-removal and harshness column bars, copied from the same frozen notebook (cell 4):
# post_rm_pal and HARSHNESS_LEVEL_MAP / HARSHNESS_PAL (harshness is a property of the method).
post_rm_pal = {False: "#CCCCCC", True: "#333333"}
POST_REMOVAL_NAMES = {False: "No post-removal", True: "Post-removal of batches"}
HARSHNESS_LEVEL_MAP = {
    "01_raw": "low", "02_median_scaling": "low", "03_limma": "low", "04_sva": "low",
    "05_combat": "medium", "06_combat_seq": "medium", "07_pycombat": "medium",
    "08_inmoose_combatseq": "medium", "09_ruv": "medium", "10_mnn": "medium",
    "11_harmony": "medium", "12_scanorama": "medium", "13_fsmvn": "medium",
    "14_qsmooth": "high", "15_fsqn_py": "high", "16_fsqn_r": "high", "17_quantile": "high",
    "18_rank": "high", "19_tdm": "high", "20_shambhala": "high", "21_harmonizr": "medium",
    "22_tmm": "low", "23_vst": "low", "24_peer_k10": "medium", "25_angel": "high",
    "26_xpn": "high", "27_dwd": "medium", "28_npn": "high", "29_combat_ref": "medium",
    "30_recombat": "medium", "31_ruv3prps": "medium", "32_deepmnn": "medium",
    "33_amdbnorm": "high", "34_arsyn": "medium", "35_dasc": "medium",
    "36_explobatch": "medium", "37_fabatch": "medium", "38_harman": "medium",
    "39_procrustes": "high",
}
HARSHNESS_PAL = {"low": "#4caf50", "medium": "#ff9800", "high": "#f44336"}
HARSHNESS_NAMES = {"low": "Low", "medium": "Medium", "high": "High"}
strat_pal = {k: mcolors.to_hex(v) for k, v in strat_pal.items()}  # one colour format everywhere

SCORE_BAR_ORDER = ["global_batch", "local_batch", "global_biology", "local_biology"]
score_bar_pal = {
    rl.PUBLICATION_NAMES["global_batch"]: "#8B4513",
    rl.PUBLICATION_NAMES["local_batch"]: "#DEB887",
    rl.PUBLICATION_NAMES["global_biology"]: "#2F4F4F",
    rl.PUBLICATION_NAMES["local_biology"]: "#B0C4DE",
    "Not in a score bar": "#DDDDDD",
}
ANNOTATION_COLUMN_NAMES = {
    "RNA_BATCH": "Batch",
    "Diagnosis_cell_type_unified": "Diagnosis",
    "none": "No annotation column",
}
annotation_column_pal = {"Batch": "#8B4513", "Diagnosis": "#2F4F4F", "No annotation column": "#808080"}
SELECTION_ROW_PAL = {"yes": "#000000", "no": "#FFFFFF"}
NA_CELL_COLOR = "#000000"  # black, as in the first article version (small decision s1)

### §0d Helpers and plot settings

In [ ]:
# Small helpers: method palette, figure saving, legends.
def build_method_palette(methods: list[str]) -> dict:
    # tab20 + tab20b with boosted saturation, as in the frozen figure notebook
    raw = (sns.color_palette("tab20", 20) + sns.color_palette("tab20b", 20))[: len(methods)]

    def boost(rgb, delta=0.15):
        h, s, v = mcolors.rgb_to_hsv(np.clip(rgb[:3], 0, 1))
        return tuple(mcolors.hsv_to_rgb([h, min(1.0, s + delta), v]))

    return {m: mcolors.to_hex(boost(c)) for m, c in zip(sorted(methods), raw)}


def save_figure(fig: plt.Figure, stem: str) -> None:
    # one file per panel: SVG + PDF + PNG (200 dpi)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


def legend_handles(palette: dict) -> list:
    # colour patches for a categorical legend
    return [mpatches.Patch(facecolor=c, edgecolor="#555555", linewidth=0.3, label=str(k)) for k, c in palette.items()]


def add_legends(fig: plt.Figure, specs: list[tuple[str, dict]], x: float | None = None,
                y_top: float = 0.98, step: float | None = None, gap_px: float = 8.0) -> float:
    """Stack categorical legends to the right of the plot, measuring each one (10 px text,
    2026-11-09): the next legend starts right below the previous one, a new column starts when
    the figure bottom is reached. x = None → right edge of the plot area. Returns the x where a
    further legend column can start. `step` is kept for old calls and not used."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_fig = fig.transFigure.inverted()
    if x is None:
        right = max(ax.get_tightbbox(renderer).x1 for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax))
        x = to_fig.transform((right + gap_px, 0))[0]
    gap = gap_px / fig.bbox.height
    y, column_right = y_top, x
    for title, palette in specs:
        legend = fig.legend(
            handles=legend_handles(palette), title=title, loc="upper left",
            bbox_to_anchor=(x, y), frameon=False, ncol=1, borderaxespad=0.0,
            handlelength=1.0, handleheight=0.9, labelspacing=0.25,
        )
        legend._legend_box.align = "left"
        box = legend.get_window_extent(renderer).transformed(to_fig)
        if box.y0 < 0 and y < y_top:  # does not fit below the previous one: new column
            x, y = column_right + gap * fig.bbox.height / fig.bbox.width, y_top
            legend.set_bbox_to_anchor((x, y), transform=fig.transFigure)
            box = legend.get_window_extent(renderer).transformed(to_fig)
        column_right = max(column_right, box.x1)
        y = box.y0 - gap
    return column_right + gap * fig.bbox.height / fig.bbox.width


GLOBAL_FONT_SIZE = 10
plt.rcParams.update({
    "font.size": GLOBAL_FONT_SIZE,
    "axes.labelsize": GLOBAL_FONT_SIZE,
    "axes.titlesize": GLOBAL_FONT_SIZE + 1,
    "xtick.labelsize": GLOBAL_FONT_SIZE - 1,
    "ytick.labelsize": GLOBAL_FONT_SIZE - 1,
    "legend.fontsize": GLOBAL_FONT_SIZE - 1,
    "pdf.fonttype": "truetype",
    "svg.fonttype": "none",
    "figure.dpi": 100,
})
sns.set_style("ticks")

In [ ]:
# Figma units (Daniil 2026-11-09, `daniil_review_261109.md`). Figma imports an SVG with
# 1 SVG pt = 4/3 px, and the figure frame (Figma node 1607-116664) is 756 × 1159 px for the
# 180 × 275 mm page. So every panel is drawn at (slot in Figma px) × 0.75 pt, and every text
# is 7.5 pt = 10 px in Figma. Legends and colour bars ("palettes") may stick out of the slot:
# only the plot area (axes with their tick labels and titles) is fitted to it.
import matplotlib.text as mtext
from matplotlib.transforms import Bbox

FIGMA_FRAME_PX = (756, 1159)
FIGMA_PAGE_MM = (180, 275)
FIGMA_PX_PER_PT = 4 / 3
FIGMA_PX_PER_MM = FIGMA_FRAME_PX[0] / FIGMA_PAGE_MM[0]  # 4.2 px per mm of the page
FIGMA_FONT_PX = 10
FIGMA_FONT_PT = FIGMA_FONT_PX / FIGMA_PX_PER_PT  # 7.5 pt
FIGMA_FIT_LOG = rl.OUTPUT_DIR / "wp14" / "figma_fit" / "nb02_reduced_panel_clustermap.tsv"
FIGMA_FIT_LOG.parent.mkdir(parents=True, exist_ok=True)

GLOBAL_FONT_SIZE = FIGMA_FONT_PT
SMALL_FONT_SIZE = FIGMA_FONT_PT
plt.rcParams.update({key: FIGMA_FONT_PT for key in (
    "font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize", "ytick.labelsize",
    "legend.fontsize", "legend.title_fontsize", "figure.titlesize")})


def mm_figsize(width_mm: float, height_mm: float) -> tuple[float, float]:
    """Figure size in inches that lands in Figma at the slot size (page mm → Figma px)."""
    to_inches = FIGMA_PX_PER_MM / FIGMA_PX_PER_PT / 72
    return width_mm * to_inches, height_mm * to_inches


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    return mm_figsize(*PANEL_SLOTS_MM[stem])


def batch_display_name(batch: str) -> str:
    """Plain batch name: underscores → spaces; microarrays drop the last "Unknown" (library
    preparation is never known for a microarray), RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def is_palette_axis(ax: plt.Axes) -> bool:
    """Colour bars and legend-only axes are palettes: Daniil moves them by hand in Figma."""
    if hasattr(ax, "_colorbar") or getattr(ax, "_figma_palette", False):
        return True
    has_data = bool(ax.lines or ax.collections or ax.images or ax.patches or ax.texts)
    return not ax.axison and not has_data


def plot_extent_px(fig: plt.Figure) -> tuple[float, float]:
    """Width and height in Figma px of the plot area (axes, tick labels, titles, texts on
    the axes, suptitle), without legends and colour bars."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_tightbbox(renderer, bbox_extra_artists=list(ax.texts))
             for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax)]
    boxes += [t.get_window_extent(renderer) for t in fig.texts if t.get_text()]
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        boxes.append(fig._suptitle.get_window_extent(renderer))
    boxes = [b for b in boxes if b is not None and b.width > 0]
    # a legend-only figure has no plot area: measure the whole figure
    union = Bbox.union(boxes) if boxes else fig.bbox
    px_per_display = 72 / fig.dpi * FIGMA_PX_PER_PT
    return union.width * px_per_display, union.height * px_per_display


def set_all_fonts(fig: plt.Figure, size: float) -> None:
    """Every text of the figure at one size (10 px in Figma)."""
    for ax in fig.axes:
        ax.tick_params(labelsize=size)
    for text in fig.findobj(mtext.Text):
        if text.get_text():
            text.set_fontsize(size)


def fit_to_slot(fig: plt.Figure, width_mm: float, height_mm: float | None,
                n_iterations: int = 8) -> None:
    """Resize the figure until its plot area is width_mm wide (and height_mm high; None keeps
    the current aspect ratio of the plot area). Fonts keep their size, so a few redraws are
    needed; fixed-aspect plots (squares) only shrink to fit."""
    target_w = width_mm * FIGMA_PX_PER_MM
    for _ in range(n_iterations):
        plot_w, plot_h = plot_extent_px(fig)
        target_h = plot_h * target_w / plot_w if height_mm is None else height_mm * FIGMA_PX_PER_MM
        if abs(plot_w - target_w) < 1 and abs(plot_h - target_h) < 1:
            break
        fig_w, fig_h = fig.get_size_inches()
        fig.set_size_inches(max(fig_w * target_w / plot_w, 0.5), max(fig_h * target_h / plot_h, 0.5))
    plot_w, plot_h = plot_extent_px(fig)
    target_h = FIGMA_FRAME_PX[1] if height_mm is None else height_mm * FIGMA_PX_PER_MM
    shrink = min(1.0, target_w / plot_w, target_h / plot_h)
    if shrink < 0.995:  # fixed-aspect plots grew in the other direction: scale the whole panel
        fig.set_size_inches(*(fig.get_size_inches() * shrink))


def log_fit(stem: str, fig: plt.Figure, slot_mm: tuple | None) -> None:
    """One row per saved panel: slot and plot area in Figma px (read by r97_figma_check.py)."""
    plot_w, plot_h = plot_extent_px(fig)
    row = pd.DataFrame([{
        "stem": stem,
        "slot_w_px": None if slot_mm is None else round(slot_mm[0] * FIGMA_PX_PER_MM, 1),
        "slot_h_px": None if slot_mm is None or slot_mm[1] is None else round(slot_mm[1] * FIGMA_PX_PER_MM, 1),
        "plot_w_px": round(plot_w, 1), "plot_h_px": round(plot_h, 1),
    }])
    old = pd.read_csv(FIGMA_FIT_LOG, sep="\t") if FIGMA_FIT_LOG.exists() else row.iloc[:0]
    pd.concat([old[old["stem"] != stem], row]).to_csv(FIGMA_FIT_LOG, sep="\t", index=False)


def save_figure(fig: plt.Figure, stem: str, slot_mm: tuple | None = None,
                keep_fonts: bool = False) -> None:
    """Fonts → 10 Figma px (unless keep_fonts: large clustermaps), plot area → slot (default:
    PANEL_SLOTS_MM[stem] when the notebook has it), then SVG + PDF + PNG (200 dpi)."""
    if slot_mm is None and stem in globals().get("PANEL_SLOTS_MM", {}):
        slot_mm = PANEL_SLOTS_MM[stem]
    if not keep_fonts:
        set_all_fonts(fig, FIGMA_FONT_PT)
    if slot_mm is not None:
        fit_to_slot(fig, *slot_mm)
    log_fit(stem, fig, slot_mm)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


## §1 Load the reduced panel and the approach metadata

Expected: 2,234 approaches × 22 panel metrics; 1,993 approaches complete on the panel; 8 metric groups in the panel
(group H, the distance ratios, is represented by its cluster mates, the two PC regressions).

In [ ]:
# Load the panel matrix (index = strat, imp, method, post_rm) and the panel description.
panel_matrix = pd.read_parquet(WP2_DIR / "reduced_panel_matrix.parquet")
reduced_panel = pd.read_csv(WP2_DIR / "reduced_panel.tsv", sep="\t").set_index("metric")
coverage_additions = pd.read_csv(WP2_DIR / "coverage_additions.tsv", sep="\t")
panel_metrics = reduced_panel.index.tolist()
panel_matrix = panel_matrix[panel_metrics]

approach_meta = panel_matrix.index.to_frame(index=False)
complete_mask = panel_matrix.notna().all(axis=1).to_numpy()
print(f"approaches: {len(panel_matrix):,}; panel metrics: {len(panel_metrics)}")
print(f"complete on the panel: {complete_mask.sum():,}; with ≥ 1 missing metric: {(~complete_mask).sum():,}")
print("metric groups:", sorted(reduced_panel["metric_group"].unique()))
assert len(panel_matrix) == 2234 and panel_matrix.index.is_unique

### §1b Plain-word labels of the panel metrics

Each panel metric gets three annotations: its metric group, its score bar (the four bars of
WP3; metrics outside a bar are labelled "Not in a score bar"), and its annotation column (batch,
diagnosis, none). Representatives chosen by the coverage rule (not by the correlation clusters)
are marked with "+" after the metric identifier.

In [ ]:
# Metric annotations in plain words.
metric_annotation = pd.DataFrame(index=panel_metrics)
metric_annotation["Metric group"] = reduced_panel["metric_group"].map(GROUP_NAMES)
metric_annotation["Score bar"] = reduced_panel["four_score_bar"].map(rl.PUBLICATION_NAMES).fillna("Not in a score bar")
metric_annotation["Annotation column"] = reduced_panel["column"].map(ANNOTATION_COLUMN_NAMES)
# Gate G1: no metric is added back any more (groups are covered through cluster mates)
coverage_added = set(coverage_additions["added"]) if "added" in coverage_additions else set()
metric_label = {m: (f"{m} +" if m in coverage_added else m) for m in panel_metrics}
assert metric_annotation.notna().all().all(), "every panel metric needs a group, bar and column"
print(metric_annotation.value_counts(["Score bar"]).to_string())
metric_annotation.to_csv(TABLES_DIR / "nb02_panel_metric_annotation.tsv", sep="\t")

### §1c The recommended approaches

The 22 approaches of the final selection (rule front, accepted at the visual check, gate G2b
closed 2026-11-07) are read from `output/wp4/selection_after_visual_check.tsv`; nothing is computed
here. Every post-removal setting is exact (the file holds the full approach key).


In [ ]:
# The 22 recommended approaches (read, not computed).
recommended = pd.read_csv(RECOMMENDED_PATH, sep="\t")
recommended = recommended[recommended["visual_check"] == "accepted"]
recommended_keys = set(zip(recommended["strat"], recommended["imp"], recommended["method"], recommended["post_rm"].astype(bool)))
approach_meta["recommended"] = [tuple(k) in recommended_keys for k in approach_meta[APPROACH_KEYS].itertuples(index=False)]
RECOMMENDED_LABEL = f"Recommended approaches (n = {len(recommended_keys)})"
print(f"recommended approaches marked: {approach_meta['recommended'].sum()} of {len(recommended_keys)}")
assert approach_meta["recommended"].sum() == len(recommended_keys)


## §2 Figure 3 — central clustermap on the reduced panel

Rows = 22 panel metrics, columns = 2,234 approaches (the orientation of the submitted Figure 3).
Values are polarity-adjusted and min-max scaled over the 2,234 approaches, so 1 is the best value
of a metric in the benchmark. Ward linkage on Euclidean distance (as submitted). Missing values are
black; for the dendrogram they are filled by the column median (switch `NA_FILL_FOR_CLUSTERING`).
The column annotation shows strategy, method, imputation, post-removal and harshness, then the 22 recommended approaches.

### §2b Naming of the clusters (data-driven)

- **Approach (column) clusters**: the Ward tree is cut into `N_COLUMN_CLUSTERS`. Scope: "same
  platform / sample type" when most approaches of the cluster come from a single-platform or
  single-sample-type strategy (RNA-seq only, FF only, FFPE only, microarrays only, Affymetrix only,
  Affymetrix extended), else "cross-platform". Quality: "good" or "poor" batch mixing by whether the
  cluster's median of the local batch-mixing panel metrics is above the median over all approaches.
- **Metric (row) clusters**: cut into `N_ROW_CLUSTERS`; named by the majority metric type of their
  metric groups (global: A, H, I, J; local: B, C, G; distribution similarity: D; bimodality and
  zero fraction: E; missing values: K), as the "Metric type" annotation of the original figure.
- Repeated names get A, B, C (e.g. "Global metrics A").

In [ ]:
# Matrix for the clustermap (metrics × approaches) and the column annotation colours.
if NA_FILL_FOR_CLUSTERING == "drop":
    heat_values = panel_matrix[complete_mask]
    heat_meta = approach_meta[complete_mask].reset_index(drop=True)
else:
    heat_values = panel_matrix
    heat_meta = approach_meta
filled_for_linkage = heat_values.fillna(heat_values.median())
column_ids = [f"approach_{i}" for i in range(len(heat_values))]

# Article method palette, copied from nb04 §0c (r98 audit 2026-11-08: building it over the 31
# analysed methods instead of the 33 of the frozen recipe gave exploBATCH the colour of ARSyN).
method_pal = {
    "01_raw": "#046cb4",
    "02_median_scaling": "#8bb3e8",
    "03_limma": "#ff7800",
    "04_sva": "#ffa852",
    "05_combat": "#14a014",
    "06_combat_seq": "#7cdf69",
    "07_pycombat": "#d60708",
    "08_inmoose_combatseq": "#ff7270",
    "09_ruv": "#864bbd",
    "10_mnn": "#b790d5",
    "11_harmony": "#8c4536",
    "12_scanorama": "#c48477",
    "13_fsmvn": "#e355b8",
    "14_qsmooth": "#f791bd",
    "15_fsqn_py": "#7f6c6c",
    "16_fsqn_r": "#c7a9a9",
    "17_quantile": "#bcbd06",
    "18_rank": "#dbdb6c",
    "19_tdm": "#00bccf",
    "20_shambhala": "#7cd5e5",
    "21_harmonizr": "#272979",
    "22_tmm": "#3a3ca3",
    "23_vst": "#4c50cf",
    "25_angel": "#7b7ede",
    "26_xpn": "#5d7927",
    "27_dwd": "#85a23a",
    "28_npn": "#adcf4c",
    "29_combat_ref": "#c7db7b",
    "31_ruv3prps": "#8c661c",
    "33_amdbnorm": "#bd971d",
    "34_arsyn": "#e7b02f",
    "36_explobatch": "#e7bf71",
    "38_harman": "#842925",
}
approach_colors = pd.DataFrame(
    {
        "Strategy": heat_meta["strat"].map(strat_pal).to_numpy(),
        "Method": heat_meta["method"].map(method_pal).to_numpy(),
        "Imputation": heat_meta["imp"].map(imp_pal).to_numpy(),
        "Post-removal": heat_meta["post_rm"].astype(bool).map(post_rm_pal).to_numpy(),
        "Harshness": heat_meta["method"].map(HARSHNESS_LEVEL_MAP).map(HARSHNESS_PAL).to_numpy(),
        RECOMMENDED_LABEL: np.where(heat_meta["recommended"], "#000000", "#FFFFFF"),
    },
    index=column_ids,
)
assert approach_colors.notna().all().all()
metric_colors = pd.DataFrame(
    {
        "Metric group": reduced_panel["metric_group"].map(group_pal),
        "Score bar": metric_annotation["Score bar"].map(score_bar_pal),
        "Annotation column": metric_annotation["Annotation column"].map(annotation_column_pal),
    }
)
assert metric_colors.notna().all().all()
metric_colors.index = [metric_label[m] for m in metric_colors.index]

col_linkage = linkage(filled_for_linkage.to_numpy(), method=CLUSTER_METHOD, metric="euclidean")
row_linkage = linkage(filled_for_linkage.T.to_numpy(), method=CLUSTER_METHOD, metric="euclidean")
# Named clusters (§2b): cut both Ward trees.
column_cluster = fcluster(col_linkage, N_COLUMN_CLUSTERS, criterion="maxclust")
row_cluster = pd.Series(fcluster(row_linkage, N_ROW_CLUSTERS, criterion="maxclust"), index=heat_values.columns)
SINGLE_TYPE_STRATEGIES = {"C_rnaseq_only", "J_ff_only", "K_ffpe_only", "F_microarray_only", "G_affymetrix_only", "H_affymetrix_extended"}
local_batch_metrics = [m for m in heat_values.columns if reduced_panel.loc[m, "four_score_bar"] == "local_batch"
                       or (metric_annotation.loc[m, "Metric group"].startswith(("B", "C")) and "RNA_BATCH" in m)]
local_batch_mean = heat_values[local_batch_metrics].mean(axis=1).to_numpy()
overall_local_median = np.nanmedian(local_batch_mean)
column_cluster_names = {}
for cluster_id in np.unique(column_cluster):
    members = column_cluster == cluster_id
    single = heat_meta.loc[members, "strat"].isin(SINGLE_TYPE_STRATEGIES).mean() > 0.5
    good = np.nanmedian(local_batch_mean[members]) >= overall_local_median
    column_cluster_names[cluster_id] = (("Same platform / sample type" if single else "Cross-platform")
                                        + (", good batch mixing" if good else ", poor batch mixing"))
# Metric type of each group (as the "Metric type" annotation of the original figure).
GROUP_TYPE = {"A": "Global metrics", "H": "Global metrics", "I": "Global metrics", "J": "Global metrics",
              "B": "Local metrics", "C": "Local metrics", "G": "Local metrics",
              "D": "Distribution similarity", "E": "Bimodality and zero fraction", "K": "Missing values"}
row_cluster_names = {}
for cluster_id in np.unique(row_cluster):
    metrics = row_cluster.index[row_cluster == cluster_id]
    types = metric_annotation.loc[metrics, "Metric group"].str[0].map(GROUP_TYPE)
    row_cluster_names[cluster_id] = types.value_counts().index[0]
for names in (column_cluster_names, row_cluster_names):  # repeated names get A, B, C (as in the original)
    counts = pd.Series(list(names.values())).value_counts()
    seen = {}
    for key in sorted(names):
        name = names[key]
        if counts[name] > 1:
            names[key] = f"{name} {'ABCDEF'[seen.get(name, 0)]}"
            seen[name] = seen.get(name, 0) + 1
column_cluster_names = {int(k): v for k, v in column_cluster_names.items()}
row_cluster_names = {int(k): v for k, v in row_cluster_names.items()}
heat_meta = heat_meta.assign(column_cluster=column_cluster, column_cluster_name=[column_cluster_names[c] for c in column_cluster])
cluster_table = pd.concat([
    heat_meta.groupby("column_cluster_name").agg(n=("strat", "size"), recommended=("recommended", "sum")).reset_index()
    .rename(columns={"column_cluster_name": "cluster"}).assign(axis="approaches"),
    row_cluster.map(row_cluster_names).rename("cluster").reset_index().rename(columns={"index": "metric"})
    .groupby("cluster").agg(n=("metric", "size"), metrics=("metric", ";".join)).reset_index().assign(axis="metrics"),
])
cluster_table.to_csv(TABLES_DIR / "nb02_fig3_clusters.tsv", sep="\t", index=False)
print(cluster_table.to_string(index=False))
heat_frame = heat_values.T.copy()
heat_frame.columns = column_ids
heat_frame.index = [metric_label[m] for m in heat_frame.index]
print(f"heat map: {heat_frame.shape[0]} metrics × {heat_frame.shape[1]:,} approaches; "
      f"missing cells: {int(heat_frame.isna().sum().sum())}")

In [ ]:
import textwrap

# Draw Figure 3: rasterized heat map, black missing cells, legends on the right.
cmap = plt.get_cmap("RdYlGn").copy()
cmap.set_bad(NA_CELL_COLOR)
fig3 = sns.clustermap(
    heat_frame,
    row_linkage=row_linkage,
    col_linkage=col_linkage,
    row_colors=metric_colors,
    col_colors=approach_colors,
    cmap=cmap, vmin=0, vmax=1, center=0.5,
    mask=heat_frame.isna(),
    figsize=mm_figsize(180, 200),
    dendrogram_ratio=(0.16, 0.12),
    colors_ratio=(0.012, 0.018),
    xticklabels=False, yticklabels=True,
    cbar_pos=(0.0, 0.86, 0.012, 0.10),
    cbar_kws={"label": "Scaled value\n(1 = best)"},
    rasterized=True,
)
fig3.ax_heatmap.set_facecolor(NA_CELL_COLOR)
fig3.ax_heatmap.set_xlabel(f"Harmonization approaches (n = {heat_frame.shape[1]:,})")
fig3.ax_heatmap.set_ylabel("")
fig3.ax_heatmap.tick_params(axis="y", labelsize=GLOBAL_FONT_SIZE - 1)
for ax in (fig3.ax_col_colors, fig3.ax_row_colors):
    for artist in ax.collections:
        artist.set_rasterized(True)
fig3.ax_col_colors.tick_params(axis="y", labelsize=GLOBAL_FONT_SIZE - 1)
fig3.ax_row_colors.tick_params(axis="x", labelsize=GLOBAL_FONT_SIZE - 1)
fig3.ax_heatmap.collections[0].set_rasterized(True)

# Named, shaded clusters (as in the original Figma figure) and black separators on the heat map.
CLUSTER_SHADES = ["#DCEBEA", "#F3DCDC", "#F4F0D8", "#E3DDF1", "#DDE6F2", "#EAEAEA"]


def spans(labels_in_order: np.ndarray) -> list[tuple]:
    # contiguous (start, end, label) runs of the reordered cluster labels
    runs, start = [], 0
    for i in range(1, len(labels_in_order) + 1):
        if i == len(labels_in_order) or labels_in_order[i] != labels_in_order[start]:
            runs.append((start, i, labels_in_order[start]))
            start = i
    return runs


col_order = np.asarray(column_cluster)[fig3.dendrogram_col.reordered_ind]
for k, (start, end, cluster_id) in enumerate(spans(col_order)):
    shade = CLUSTER_SHADES[k % len(CLUSTER_SHADES)]
    fig3.ax_col_dendrogram.axvspan(start * 10, end * 10, color=shade, alpha=0.8, zorder=0, linewidth=0)
    fig3.ax_col_dendrogram.text((start + end) * 5, fig3.ax_col_dendrogram.get_ylim()[1] * 0.97,
                                textwrap.fill(column_cluster_names[cluster_id], 16), ha="center", va="top",
                                fontsize=GLOBAL_FONT_SIZE - 1, fontstyle="italic", color="#444444")
    if start:
        fig3.ax_heatmap.axvline(start, color="black", linewidth=1.0)
row_order = row_cluster.to_numpy()[fig3.dendrogram_row.reordered_ind]
for k, (start, end, cluster_id) in enumerate(spans(row_order)):
    shade = CLUSTER_SHADES[(k + 2) % len(CLUSTER_SHADES)]
    fig3.ax_row_dendrogram.axhspan(start * 10, end * 10, color=shade, alpha=0.8, zorder=0, linewidth=0)
    fig3.ax_row_dendrogram.text(fig3.ax_row_dendrogram.get_xlim()[0] * 0.98, (start + end) * 5,
                                textwrap.fill(row_cluster_names[cluster_id], 11), ha="left", va="center", fontsize=GLOBAL_FONT_SIZE - 1,
                                fontstyle="italic", color="#444444")
    if start:
        fig3.ax_heatmap.axhline(start, color="black", linewidth=1.0)

used_methods = sorted(heat_meta["method"].unique())
set_all_fonts(fig3.fig, FIGMA_FONT_PT)
FIG3_SLOT_MM = (180, 200)  # plot area; legends to its right (Daniil moves them in Figma)
fit_to_slot(fig3.fig, *FIG3_SLOT_MM)
next_column = add_legends(
    fig3.fig,
    [
        ("Strategy", {rl.STRATEGY_DISPLAY_NAMES[s]: c for s, c in strat_pal.items() if s in set(heat_meta["strat"])}),
        ("Imputation", {IMPUTATION_NAMES[k]: c for k, c in imp_pal.items()}),
        ("Post-removal", {POST_REMOVAL_NAMES[k]: c for k, c in post_rm_pal.items()}),
        ("Harshness", {HARSHNESS_NAMES[k]: c for k, c in HARSHNESS_PAL.items()}),
        ("Metric group", {g: group_pal[k] for k, g in GROUP_NAMES.items()}),
        ("Score bar", score_bar_pal),
        ("Annotation column", annotation_column_pal),
    ],
)
add_legends(
    fig3.fig,
    [("Method", {rl.method_display_name(m): method_pal[m] for m in used_methods})],
    x=next_column,
)
save_figure(fig3.fig, "fig3_reduced_panel_clustermap", slot_mm=FIG3_SLOT_MM)
plt.show()

In [ ]:
# Save the column order of Figure 3 (approach metadata in dendrogram order) for traceability.
fig3_order = heat_meta.iloc[fig3.dendrogram_col.reordered_ind].reset_index(drop=True)
fig3_order["method_display"] = fig3_order["method"].map(rl.method_display_name)
fig3_order.to_csv(TABLES_DIR / "nb02_fig3_column_order.tsv", sep="\t", index=False)
print(fig3_order[fig3_order["recommended"]][APPROACH_KEYS + ["column_cluster_name"]].to_string())

## §3 Supplementary Figure S4 — metric availability by strategy × imputation

Share of approaches (analysis set, 2,234) on which every metric is defined: the 22-metric
reduced panel against the 87 published scoring metrics. Dropping the platform / source / cohort
variants makes availability nearly equal across strategies (expected 1,993 vs 1,354 complete
approaches overall).

In [ ]:
# Completeness on the panel and on the 87 published scoring metrics, per strategy × imputation.
polarity = pd.read_excel(SUPP_FILE_2_PATH, sheet_name="Metric_polarity").set_index("Metric")["Polarity"]
supp_file_3 = pd.read_csv(SUPP_FILE_3_PATH)
supp_file_3["post_rm"] = supp_file_3["post_rm"].astype(bool)
analysis_set = supp_file_3[supp_file_3["pct_samples_allNA"] < 5].copy()
scoring_metrics = [m for m in polarity.index[polarity != 0] if m in analysis_set.columns]
assert len(scoring_metrics) == 87 and len(analysis_set) == 2234

analysis_set["complete_panel"] = analysis_set[panel_metrics].notna().all(axis=1)
analysis_set["complete_87"] = analysis_set[scoring_metrics].notna().all(axis=1)
availability = (
    analysis_set.groupby(["strat", "imp"])
    .agg(n=("method", "size"), complete_panel=("complete_panel", "sum"), complete_87=("complete_87", "sum"))
    .reset_index()
)
availability["pct_panel"] = 100 * availability["complete_panel"] / availability["n"]
availability["pct_87"] = 100 * availability["complete_87"] / availability["n"]
availability.to_csv(TABLES_DIR / "nb02_figS4_availability.tsv", sep="\t", index=False)
print(f"complete on the panel: {analysis_set['complete_panel'].sum():,}; on all 87: {analysis_set['complete_87'].sum():,}")

In [ ]:
# Draw S4: two heat maps side by side (panel | 87 metrics); each cell shows the count of
# approaches with every metric defined / all approaches (no %, compact; Daniil 2026-11-09).
strategy_order = [s for s in rl.STRATEGY_DISPLAY_NAMES if s in set(availability["strat"])]
imputation_order = ["strict", "knn", "softimpute"]
panels = [("pct_panel", "complete_panel", f"Reduced panel ({len(panel_metrics)} metrics)"),
          ("pct_87", "complete_87", "All 87 scoring metrics")]
figS4, axes = plt.subplots(1, 2, figsize=mm_figsize(130, 62), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, (pct_col, n_col, title) in zip(axes, panels):
    pct = availability.pivot(index="strat", columns="imp", values=pct_col).loc[strategy_order, imputation_order]
    counts = availability.pivot(index="strat", columns="imp", values=n_col).loc[strategy_order, imputation_order]
    totals = availability.pivot(index="strat", columns="imp", values="n").loc[strategy_order, imputation_order]
    labels = counts.astype(int).astype(str) + "/" + totals.astype(int).astype(str)
    sns.heatmap(pct, ax=ax, cmap="YlGnBu", vmin=0, vmax=100, annot=labels, fmt="",
                annot_kws={"size": GLOBAL_FONT_SIZE}, cbar=ax is axes[-1],
                cbar_kws={"label": "Approaches with every metric defined (%)"}, linewidths=0.4, linecolor="white")
    ax.set_title(title, loc="left")
    ax.set_xticklabels([IMPUTATION_NAMES[i] for i in imputation_order], rotation=0)
    ax.set_xlabel("Imputation")
    ax.set_ylabel("")
axes[0].set_yticklabels([rl.STRATEGY_DISPLAY_NAMES[s] for s in strategy_order], rotation=0)
save_figure(figS4, "figS4_metric_availability", slot_mm=(130, 62))
plt.show()

## §4 Supplementary Figure S5 — the 22 panel metrics inside the 87 scoring metrics

**Daniil, 2026-11-08 (`gates_review_261102.md`, request after §10.6):** replace S5 by the
Spearman cross-correlation clustermap of all **87 scoring metrics** (as Supplementary Figure 7 of the
submitted article, Figma node 401-3), with one more colour bar: whether the metric is in the final
22-metric panel (★ on its label). The metric colour bars are those of Figure 3 (metric group, score
bar, annotation column), plus the metric type of the original figure.

- Values: Spearman ρ between the polarity-adjusted metrics over the 2,234 approaches (pairwise
  complete), so a positive ρ means that two metrics agree on which approach is better.
- Tree: complete linkage on 1 − |ρ|, the rule WP2 used to build the panel (cut at |ρ| ≥ 0.8,
  dashed line). The 87 include the platform, RNA-seq source, cohort and tumour / normal variants
  that the panel rule excluded by design (it took only batch, diagnosis or no annotation column).
- Medoid check (table `output/wp9/nb02_figS5_87_metric_clusters.tsv`): the tree is cut at |ρ| ≥ 0.8;
  for every cluster the medoid is the metric with the highest mean |ρ| to its cluster mates. The
  table says, for each panel metric, its cluster size, whether it is the medoid, and its mean |ρ| to
  the cluster against that of the medoid. This tests Daniil's reading "the 22 are the cluster
  centres of the 87"; the answer is reported as it comes out.

The former S5 (33 candidate metrics) is no longer drawn; its data stay in `output/wp2/`.

In [ ]:
# Polarity-adjusted 87 scoring metrics over the 2,234 approaches (analysis set of §3).
polarity_table = rl.load_metric_polarity()
scoring_frame = analysis_set[scoring_metrics].astype(float) * polarity.loc[scoring_metrics].to_numpy()
rho_87 = scoring_frame.corr(method="spearman", min_periods=100)
assert rho_87.shape == (87, 87) and set(panel_metrics) <= set(scoring_metrics)
distance_87 = (1.0 - rho_87.abs().fillna(0.0)).clip(lower=0.0)
np.fill_diagonal(distance_87.values, 0.0)
tree_87 = linkage(squareform(distance_87.values, checks=False), method="complete")
cluster_87 = pd.Series(fcluster(tree_87, PANEL_CUT, criterion="distance"), index=scoring_metrics)

# Medoid check: for each |ρ| ≥ 0.8 cluster, the metric with the highest mean |ρ| to its mates.
medoid_rows = []
for cluster_id, members in cluster_87.groupby(cluster_87).groups.items():
    members = list(members)
    mean_abs = (rho_87.loc[members, members].abs().sum(axis=1) - 1) / max(len(members) - 1, 1)
    medoid = mean_abs.idxmax() if len(members) > 1 else members[0]
    for metric in members:
        medoid_rows.append({"metric": metric, "cluster": int(cluster_id), "cluster_size": len(members),
                            "in_panel": metric in panel_metrics, "is_medoid": metric == medoid,
                            "medoid": medoid,
                            "mean_abs_rho_to_cluster": round(float(mean_abs[metric]), 3) if len(members) > 1 else np.nan,
                            "medoid_mean_abs_rho": round(float(mean_abs[medoid]), 3) if len(members) > 1 else np.nan,
                            "panel_metrics_in_cluster": int(sum(m in panel_metrics for m in members))})
medoid_table = pd.DataFrame(medoid_rows).sort_values(["cluster", "is_medoid"], ascending=[True, False])
medoid_table.to_csv(TABLES_DIR / "nb02_figS5_87_metric_clusters.tsv", sep="\t", index=False)
panel_rows = medoid_table[medoid_table["in_panel"]]
print(f"87 metrics → {cluster_87.nunique()} clusters at |ρ| ≥ 0.8; "
      f"{(medoid_table.groupby('cluster')['panel_metrics_in_cluster'].first() > 0).sum()} clusters hold a panel metric")
print(f"panel metrics that are the medoid (or alone) in their cluster: {int(panel_rows['is_medoid'].sum())} of {len(panel_rows)}")
print(panel_rows[["metric", "cluster_size", "is_medoid", "medoid", "mean_abs_rho_to_cluster", "medoid_mean_abs_rho"]].to_string(index=False))

In [ ]:
# Draw the new S5: signed polarity-adjusted ρ, complete-linkage tree on 1 − |ρ|, Figure 3 colour bars.
COLUMN_OF = {"RNA_BATCH": "Batch", "Diagnosis_cell_type_unified": "Diagnosis", "PLATFORM_RNA": "Platform",
             "RNASEQ_SOURCE": "RNA-seq source", "COHORT_LABEL": "Cohort", "TUMOR_NORMAL": "Tumour / normal"}
annotation_column_pal_87 = {**annotation_column_pal, "Platform": "#C49A6C", "RNA-seq source": "#E3C9A8",
                            "Cohort": "#A0522D", "Tumour / normal": "#6B8E9F"}
METRIC_TYPE_PAL = {"Global": "#4C72B0", "Local": "#DD8452", "Distributional similarity": "#55A868", "Other": "#B0B0B0"}
NOT_IN_PANEL = "Not in the panel"
s5_annotation = pd.DataFrame(index=scoring_metrics)
s5_annotation["In the 22-metric panel"] = ["yes" if m in panel_metrics else "no" for m in scoring_metrics]
s5_annotation["Metric group"] = polarity_table.loc[scoring_metrics, "Metric group"].map(GROUP_NAMES)
s5_annotation["Score bar"] = [metric_annotation.loc[m, "Score bar"] if m in panel_metrics else NOT_IN_PANEL
                              for m in scoring_metrics]
s5_annotation["Annotation column"] = [next((name for col, name in COLUMN_OF.items() if col in m), "No annotation column")
                                      for m in scoring_metrics]
s5_annotation["Metric type"] = polarity_table.loc[scoring_metrics, "Metric type"]
assert s5_annotation.notna().all().all()
score_bar_pal_87 = {**score_bar_pal, NOT_IN_PANEL: "#F2F2F2"}
group_pal_by_name = {GROUP_NAMES[k]: group_pal[k] for k in GROUP_NAMES}
s5_colors = pd.DataFrame({
    "In the 22-metric panel": s5_annotation["In the 22-metric panel"].map(SELECTION_ROW_PAL),
    "Metric group": s5_annotation["Metric group"].map(group_pal_by_name),
    "Score bar": s5_annotation["Score bar"].map(score_bar_pal_87),
    "Annotation column": s5_annotation["Annotation column"].map(annotation_column_pal_87),
    "Metric type": s5_annotation["Metric type"].map(METRIC_TYPE_PAL),
})
assert s5_colors.notna().all().all()
s5_labels = [f"{m} ★" if m in panel_metrics else m for m in scoring_metrics]
s5_colors.index = s5_labels
s5_frame = rho_87.copy()
s5_frame.index = s5_labels
s5_frame.columns = s5_labels
figS5 = sns.clustermap(
    s5_frame, row_linkage=tree_87, col_linkage=tree_87,
    row_colors=s5_colors, col_colors=s5_colors[["In the 22-metric panel", "Metric group"]],
    cmap="RdBu_r", vmin=-1, vmax=1, center=0,
    figsize=(17, 16.5), dendrogram_ratio=(0.10, 0.08), colors_ratio=(0.012, 0.012),
    xticklabels=False, yticklabels=True,
    cbar_pos=(0.02, 0.90, 0.012, 0.07), cbar_kws={"label": "Spearman ρ\n(polarity-adjusted)"},
    rasterized=True,
)
S5_ROW_LABEL_PT = 4.5  # 6 px in Figma: 87 rows on ~170 mm
figS5.ax_heatmap.tick_params(axis="y", labelsize=S5_ROW_LABEL_PT)
for label in figS5.ax_heatmap.get_yticklabels():
    if label.get_text().endswith("★"):
        label.set_fontweight("bold")
figS5.ax_row_colors.tick_params(axis="x", labelsize=GLOBAL_FONT_SIZE - 2)
figS5.ax_row_dendrogram.axvline(PANEL_CUT, color="#333333", linestyle="--", linewidth=1)
figS5.ax_col_dendrogram.axhline(PANEL_CUT, color="#333333", linestyle="--", linewidth=1)
used = lambda column: set(s5_annotation[column])
set_all_fonts(figS5.fig, FIGMA_FONT_PT)
figS5.ax_heatmap.tick_params(axis="y", labelsize=S5_ROW_LABEL_PT)
fit_to_slot(figS5.fig, 180, None)
add_legends(
    figS5.fig,
    [("In the 22-metric panel (★)", {"Yes": SELECTION_ROW_PAL["yes"], "No": SELECTION_ROW_PAL["no"]}),
     ("Metric group", {g: c for g, c in group_pal_by_name.items() if g in used("Metric group")}),
     ("Score bar", {b: c for b, c in score_bar_pal_87.items() if b in used("Score bar")}),
     ("Annotation column", {a: c for a, c in annotation_column_pal_87.items() if a in used("Annotation column")}),
     ("Metric type", {t: c for t, c in METRIC_TYPE_PAL.items() if t in used("Metric type")}),
     ("Tree", {"dashed line: |ρ| = 0.8 cut": "#FFFFFF"})],

)
# 87 metric names: the only panel with small row labels (Daniil 2026-11-09)
save_figure(figS5.fig, "figS5_metric_correlation_clusters", slot_mm=(180, None), keep_fonts=True)
plt.show()

## §5 Supplementary Figure S6 — approach × approach correlation on the reduced panel

Spearman ρ between approaches over the 22 panel metrics (scaled values), as the submitted
Supplementary Figure 9 did over the 87 metrics. Only the 1,993 approaches complete on the panel
are used, so that every correlation is computed on the same 22 metrics. Ward linkage; row and
column colours: strategy, method, imputation.

In [ ]:
# Spearman correlation between complete approaches (rank the metrics within each approach).
complete_panel = panel_matrix[complete_mask]
complete_meta = approach_meta[complete_mask].reset_index(drop=True)
approach_rho = np.corrcoef(complete_panel.rank(axis=1).to_numpy())
approach_rho = pd.DataFrame(approach_rho)
print(f"approach correlation: {approach_rho.shape}; median ρ {np.median(approach_rho.values[np.triu_indices(len(approach_rho), 1)]):.3f}")
s6_colors = pd.DataFrame(
    {
        "Strategy": complete_meta["strat"].map(strat_pal),
        "Method": complete_meta["method"].map(method_pal),
        "Imputation": complete_meta["imp"].map(imp_pal),
    }
)

In [ ]:
# Draw S6 (rasterized; no tick labels for 1,993 approaches).
figS6 = sns.clustermap(
    approach_rho, method=CLUSTER_METHOD, metric="euclidean",
    row_colors=s6_colors, col_colors=s6_colors,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    figsize=(12, 12), dendrogram_ratio=(0.08, 0.08), colors_ratio=0.015,
    xticklabels=False, yticklabels=False,
    cbar_pos=(0.02, 0.86, 0.015, 0.10), cbar_kws={"label": "Spearman ρ"},
    rasterized=True,
)
figS6.ax_heatmap.collections[0].set_rasterized(True)
for ax in (figS6.ax_row_colors, figS6.ax_col_colors):
    for artist in ax.collections:
        artist.set_rasterized(True)
    ax.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
figS6.ax_heatmap.set_xlabel(f"Approaches complete on the reduced panel (n = {len(approach_rho):,})")
set_all_fonts(figS6.fig, FIGMA_FONT_PT)
fit_to_slot(figS6.fig, 180, None)
next_column = add_legends(
    figS6.fig,
    [("Strategy", {rl.STRATEGY_DISPLAY_NAMES[s]: c for s, c in strat_pal.items() if s in set(complete_meta["strat"])}),
     ("Imputation", {IMPUTATION_NAMES[k]: c for k, c in imp_pal.items()})],

)
add_legends(
    figS6.fig,
    [("Method", {rl.method_display_name(m): method_pal[m] for m in sorted(complete_meta["method"].unique())})],
    x=next_column,
)
save_figure(figS6.fig, "figS6_approach_correlation", slot_mm=(180, None))
plt.show()

## §6 Summary

In [ ]:
# Summary of what was drawn.
summary = {
    "approaches_in_fig3": int(heat_frame.shape[1]),
    "panel_metrics": len(panel_metrics),
    "complete_on_panel": int(analysis_set["complete_panel"].sum()),
    "complete_on_87": int(analysis_set["complete_87"].sum()),
    "recommended_columns_marked": int(heat_meta["recommended"].sum()),
    "column_clusters": column_cluster_names,
    "row_clusters": {int(k): v for k, v in row_cluster_names.items()},
    "approaches_in_figS6": int(len(approach_rho)),
}
(TABLES_DIR / "nb02_summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))

## §7 Notes (gate G4 answered 2026-11-07)

1. Figure 3 marks the 22 recommended approaches; no mark of the submitted version.
2. Cluster names are data-driven (§2b) and drawn on shaded bands; Daniil may rename them in Figma
   when he assembles the page. The cluster table is `output/wp9/nb02_fig3_clusters.tsv`.
3. Missing values: 241 approaches miss ≥ 1 panel metric (black cells; the dendrogram uses
   median-filled values; `NA_FILL_FOR_CLUSTERING = "drop"` clusters only the 1,993 complete ones).
